In [1]:
import selenium.webdriver
from selenium.webdriver.common.by import By

from selenium.webdriver.support.ui import WebDriverWait

from urllib import request
import pandas as pd
import re
from tqdm import tqdm

Ce fichier scrap les discours politiques

In [2]:
dep=pd.read_csv("C:\dalas\projet\deputes\deputes.csv")

In [3]:
discours={}
drive=selenium.webdriver.Firefox()

In [4]:
#from collections import defaultdict
import os
def read(politic):
    convert=politic.replace(' ','+')
    page=0
    while True:
        http=f'https://www.vie-publique.fr/discours/recherche?search_api_fulltext_discours=&sort_by=field_date_prononciation_discour_1&field_intervenant_title={convert}&field_intervenant_qualite=&field_date_prononciation_discour_interval%5Bmin%5D=2000-01-01&field_date_prononciation_discour_interval%5Bmax%5D=2025-02-16&field_type_de_document[9346]=9346&field_type_de_document[9350]=9350&field_type_de_document[9351]=9351&field_type_de_document[9364]=9364&page={page}'
        drive.get(http)
        page+=1
        links=[]

        if page==1:
            try:
                a=drive.find_element(By.CSS_SELECTOR,'div.vp-results-amount-select').find_element(By.CSS_SELECTOR,'h2').text
                num=""
                i=0
                while a[i].isalnum():
                    num+=a[i]
                    i+=1
                num=int(num)
                
                
                step=1
                if num>=100:
                    step=2
                if num>=300:
                    step=3
            except:
                break

        tab=drive.find_elements(By.CSS_SELECTOR,'div.views-row')

        for i in tab:#range(0,len(tab),step):
            text=i.find_element(By.CSS_SELECTOR,'a').get_attribute('href')
          #  label=i.find_element(By.CSS_SELECTOR,'span.field__item').text
            if not('debat' in text):
                links.append(text)
        if len(links)==0:
            break
        os.makedirs(f"discours\{politic}",exist_ok=True)
        for t in range(0,len(links),step):
            
                text=links[t]
            #    print(text)
                try:
                    drive.get(text)
                except:
                    continue
                intervenant=drive.find_element(By.CSS_SELECTOR,'ul.line-intervenant').find_elements(By.CSS_SELECTOR,'li')
                if len(intervenant)==1: #and convert in intervenant[0].find_element(By.CSS_SELECTOR,'a').text:
                   # print('good')
                    #speach=""
                    try:
                        date=drive.find_element(By.CSS_SELECTOR,'span.field.field--name-field-date-prononciation-discour.field--type-datetime.field--label-hidden.field__item').text
                        #speach+=date.text.strip() + ' ' + convert+ ' '
                    except:
                        date=drive.find_element(By.CSS_SELECTOR,'div.vp-discours-details').find_element(By.CSS_SELECTOR,'p').text[11:]
                        #speach+=date.text.strip() + ' ' + convert+ ' '
                   #     speach+=' ' + convert+ ' '
                    try:
                    
                        content=drive.find_element(By.CSS_SELECTOR,'div.field.field--name-field-texte-integral.field--type-text-long.field--label-hidden.field__item').text.strip()
                        #speach+=content
                        
                        #clean_content=re.split(r'[^a-zA-Zà-ÿÀ-Ÿ]+',content)
                        # del(content)
                        #interest=[i.lower() for i in clean_content if len(i)>3]
                        #del clean_content
                        #occurence=defaultdict(int)
                        #for i in clean_content:
                            #   if len(i)>3:
                            #      occurence[i.lower()]+=1
                        if len(content)>0:
                            dir=f"discours\{politic}\{date.strip()}.txt"
                            
                            with open(dir, "a", encoding="utf-8") as f:
                                f.write(content)
                            f.close()
                     
                        #speachs.append((date.text.strip(),occurence))
                    except:
                         None
        

In [7]:
# save = open(f'discours_save.txt', 'a') 
# save.write('{')
# save.close() 
for i in tqdm(list(dep['nom'].unique())[409+688+4:]):
   # print(i)
    disc=read(i)
   # print(disc)
   # j=list(dep[dep['nom']==i]['groupe_sigle'].unique())
    # if len(disc)>0:
    #     save = open(f'occurence.txt', 'a',encoding='utf-8') 
    #     save.write(f"{i}:{str(disc)},")
    #     save.close() 
# save = open(f'discours_save.txt', 'a')    
# save.write('}')
# save.close() 

  0%|          | 0/612 [00:00<?, ?it/s]

100%|██████████| 612/612 [1:19:47<00:00,  7.82s/it]   
